# 012 · Load bronze with COPY INTO
Loads landing JSON files into `<catalog>.bronze.complaints_bronze`. Idempotent: already-loaded files are skipped.

### Load
Creates a placeholder table (schema inferred via `mergeSchema`), loads `*.json` only, and adds `source_file` and `loaded_at`.
First run (11/23): 7,329,948 rows. Re-runs load 0.

In [0]:
dbutils.widgets.text("catalog", "cfpb_dev")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = "bronze"
TABLE   = f"{CATALOG}.{SCHEMA}.complaints_bronze"
LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/landing/complaints"

spark.sql(f"CREATE TABLE IF NOT EXISTS {TABLE}")

display(spark.sql(f"""
  COPY INTO {TABLE}
  FROM (
    SELECT *,
           _metadata.file_name AS source_file,
           current_timestamp()  AS loaded_at
    FROM '{LANDING}'
  )
  FILEFORMAT = JSON
  PATTERN = '*.json'
  FORMAT_OPTIONS ('mergeSchema' = 'true')
  COPY_OPTIONS ('mergeSchema' = 'true')
"""))

### Validate
Rows should equal distinct `complaint_id`s, with 365 source files (as of 11/23/2026).

In [0]:
display(spark.sql(f"""
  SELECT COUNT(*) AS rows,
         COUNT(DISTINCT complaint_id) AS ids,
         COUNT(DISTINCT source_file) AS files
  FROM {TABLE}
"""))